# 실습 1주차: 데이터를 행렬로 — 벡터와 행렬 다루기

> **시나리오 — 오늘 할 일**
>
>
> 딥러닝 모형에 들어가는 것은 **언제나 숫자 행렬**이다.
> 표든, 사진이든, 문장이든 예외가 없다.
>
> $$\text{표} \;\to\; (n, p) \qquad \text{사진} \;\to\; (H, W, C) \qquad \text{문장} \;\to\; (T,)$$
>
> 오늘은 **행렬을 자유롭게 다루는 연습**을 하고,
> 이 수업에서 다룰 **세 종류의 데이터를 각각 행렬로 만들어 본다.**
> 마지막에 행렬 곱 한 번이 곧 **예측**이라는 것을 확인한다.
>
> - **대응 이론**: [Ch01 들어가기: 데이터와 모형](ch01.qmd)
> - 코드는 완성되어 있다. **직접 해보기** 칸은 스스로 채운 뒤 아래 정답과 맞춰 본다. 채점하지 않는다.


> **오늘 익히는 것**
>
>
> | 하고 싶은 일 | 도구 |
> |------|------|
> | 벡터·행렬 만들기, 모양 확인 | `torch.tensor`, `.shape`, `.dtype` |
> | 행(데이터 포인트)·열(변수) 꺼내기 | 인덱싱, 슬라이싱, 불리언 마스크 |
> | 변수별 / 데이터별 통계 | `dim=0`, `dim=1` |
> | 정답을 세로로 세우기 | `unsqueeze(1)` |
> | 모양이 달라도 계산하기 | 브로드캐스팅 |
> | **여러 데이터를 행렬로** | 표 · 사진 · 문장 |
> | 예측을 한 번에 | 행렬 곱 `X @ w` |

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)
print('torch', torch.__version__)

---

# 1. 벡터와 행렬

## 1-1. 1차원 = 벡터

In [ ]:
v = torch.tensor([3.0, 1.0, 4.0, 1.0, 5.0])

print('값     :', v)
print('shape  :', v.shape, '  ← 길이 5인 1차원')
print('차원 수 :', v.dim())
print('dtype  :', v.dtype)
print('개수   :', v.numel())

## 1-2. 2차원 = 행렬

In [ ]:
M = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print(M)
print('shape  :', M.shape, '  ← (행, 열) = (2, 3)')
print('차원 수 :', M.dim())
print('개수   :', M.numel())

> **`shape` 을 읽는 법**
>
>
> `torch.Size([2, 3])` 은 **행 2개, 열 3개**를 뜻한다.
> 앞으로 나올 모든 텐서는 이렇게 읽는다.
>
> $$(2,\ 3) \;\to\; \text{행렬} \qquad
> (333,\ 4) \;\to\; \text{펭귄 333마리} \times \text{변수 4개} \qquad
> (64,\ 3,\ 32,\ 32) \;\to\; \text{사진 64장}$$
>
> 막히면 **`.shape` 을 찍는다.** 딥러닝 에러의 대부분이 모양 문제다.


## 1-3. `dtype` — 딥러닝은 `float32`

In [ ]:
a = torch.tensor([1, 2, 3])                 # 정수로 만들면
b = torch.tensor([1.0, 2.0, 3.0])           # 소수점을 찍으면
print('a :', a.dtype)
print('b :', b.dtype)

arr = np.array([1.0, 2.0, 3.0])             # numpy의 기본은 float64
print('\nnumpy 기본        :', arr.dtype)
print('그대로 텐서로      :', torch.tensor(arr).dtype)
print('float32로 만들기   :', torch.tensor(arr, dtype=torch.float32).dtype)

> **`float64` 를 넣으면 에러가 난다**
>
>
> PyTorch 신경망은 **`float32`** 로 계산한다. numpy 배열을 그대로 넣으면
> `expected Float but found Double` 이 뜬다. 데이터를 만들 때 항상 `float32` 로 맞춘다.


> **직접 해보기 ① — 행렬 만들기**
>
>
> 3행 4열짜리 행렬 `A` 를 `float32` 로 만드시오. 값은 아무거나 좋다.

In [ ]:
# ✏️ 직접 채워 보세요
A = None            # ← 여기를 채우세요

assert A is not None and A.shape == (3, 4), f'shape을 확인하세요: {A.shape}'
assert A.dtype == torch.float32, f'dtype을 확인하세요: {A.dtype}'
print('통과', A.shape, A.dtype)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
A = torch.tensor([[1., 2., 3., 4.],
                  [5., 6., 7., 8.],
                  [9., 10., 11., 12.]])
print(A)
print(A.shape, A.dtype)

---

# 2. 행과 열 — 데이터 포인트와 변수

## 2-1. 표를 행렬로

In [ ]:
URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
df = pd.read_csv(URL).dropna().reset_index(drop=True)

cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
X = torch.tensor(df[cols].to_numpy(dtype='float32'))

print('shape :', X.shape, '  ← (n, p) = (펭귄 수, 변수 수)')
print('dtype :', X.dtype)
print('\n앞 3줄:\n', X[:3])

> **`(n, p)` — 이 수업의 기본 모양**
>
>
> - **행** 하나가 **데이터 포인트** 하나 (펭귄 한 마리)
> - **열** 하나가 **변수** 하나 (부리 길이)
>
> Ch01에서 말한 **정형 데이터**가 코드에서는 이 모양이다.


## 2-2. 한 마리, 한 변수 꺼내기

In [ ]:
print('0번 펭귄  X[0]     :', X[0], '  shape', X[0].shape)
print('날개 길이 X[:, 2]  :', X[:5, 2], '...  shape', X[:, 2].shape)
print('0번의 날개 X[0, 2] :', float(X[0, 2]))

In [ ]:
print('앞 5마리          X[:5].shape      :', X[:5].shape)
print('부리 두 변수만     X[:, :2].shape   :', X[:, :2].shape)
print('10~14번, 뒤 두 변수 X[10:15, 2:].shape:', X[10:15, 2:].shape)

## 2-3. 조건으로 골라내기 — 불리언 마스크

In [ ]:
mask = X[:, 3] > 5000                      # 몸무게 5kg 초과
print('mask shape :', mask.shape, mask.dtype)
print('앞 10개    :', mask[:10])
print('해당 마리 수:', int(mask.sum()))
print('\n골라낸 행렬:', X[mask].shape)

In [ ]:
# 조건을 조합할 수도 있다
big_and_long = (X[:, 3] > 5000) & (X[:, 2] > 220)
print('무겁고 날개도 긴 펭귄:', int(big_and_long.sum()), '마리')
print(X[big_and_long][:3])

## 2-4. 정답 `y` 는 세로로 세운다

In [ ]:
y = torch.tensor(df['body_mass_g'].to_numpy(dtype='float32'))
print('y            :', y.shape, ' ← 1차원')
print('y.unsqueeze(1):', y.unsqueeze(1).shape, ' ← 세로로 세운 (n, 1)')

> **`(n,)` 과 `(n, 1)` 을 섞지 않는다**
>
>
> 모형의 출력이 `(n, 1)` 이면 정답도 `(n, 1)` 이어야 손실이 제대로 계산된다.
> 섞으면 **에러 없이 엉뚱한 값**이 나오기도 한다. `unsqueeze(1)` 이 축 하나를 끼워 넣어 준다.


> **직접 해보기 ② — 조건으로 고르기**
>
>
> 부리 길이(`X[:, 0]`)가 45mm 이상인 펭귄만 골라 `X_long` 에 담고, 몇 마리인지 세시오.

In [ ]:
# ✏️ 직접 채워 보세요
X_long = None            # ← 여기를 채우세요

assert X_long is not None and X_long.shape[1] == 4, '모양을 확인하세요'
print('통과', X_long.shape[0], '마리')

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
X_long = X[X[:, 0] >= 45.0]
print(X_long.shape[0], '마리   shape', tuple(X_long.shape))

---

# 3. 축(`dim`) — 어느 방향으로 계산하나

In [ ]:
print('전체 평균        :', float(X.mean()))
print('변수별 평균 dim=0:', X.mean(dim=0).numpy().round(2), ' shape', X.mean(dim=0).shape)
print('마리별 평균 dim=1:', X.mean(dim=1)[:5].numpy().round(2), '... shape', X.mean(dim=1).shape)

> **축을 외우는 법**
>
>
> **`dim=k` 는 "k번째 축을 없앤다"** 로 읽으면 헷갈리지 않는다.
>
> `X` 가 `(333, 4)` 일 때
> - `X.mean(dim=0)` → 0번 축(333)이 사라져 `(4,)` — **변수별** 평균
> - `X.mean(dim=1)` → 1번 축(4)이 사라져 `(333,)` — **데이터별** 평균
>
> 우리가 거의 항상 쓰는 것은 **`dim=0`** 이다. 변수마다 통계를 내기 때문이다.

In [ ]:
print('변수별 최솟값:', X.min(dim=0).values.numpy().round(1))
print('변수별 최댓값:', X.max(dim=0).values.numpy().round(1))
print('변수별 표준편차:', X.std(dim=0, unbiased=False).numpy().round(2))

> **직접 해보기 ③ — 변수별 범위**
>
>
> 각 변수의 **최댓값 − 최솟값**을 구해 `spread` 에 담으시오. shape은 `(4,)` 여야 한다.

In [ ]:
# ✏️ 직접 채워 보세요
spread = None            # ← 여기를 채우세요

assert spread is not None and spread.shape == (4,), f'shape 확인: {spread.shape}'
print('통과', spread.numpy().round(1))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
spread = X.max(dim=0).values - X.min(dim=0).values
print(pd.DataFrame({'변수': cols, '범위': spread.numpy().round(1)}).to_string(index=False))

몸무게만 범위가 압도적으로 크다. 이것이 나중에 문제가 된다 — 4주차에서 다룬다.

---

# 4. 브로드캐스팅 — 모양이 달라도 계산된다

In [ ]:
M = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
row = torch.tensor([10.0, 20.0, 30.0])

print('M    :', M.shape)
print('row  :', row.shape)
print('M+row:', (M + row).shape)
print(M + row, '  ← row가 모든 행에 더해졌다')

> **브로드캐스팅 규칙**
>
>
> 두 텐서의 shape을 **뒤에서부터** 비교한다.
>
> - 크기가 **같으면** 통과
> - 한쪽이 **1이면** 그쪽을 늘려서 맞춘다
> - 둘 다 아니면 **에러**
>
> ```
> M    (2, 3)
> row     (3)   →  (1, 3) 로 보고 → 2번 반복 → (2, 3)
> ```


## 4-1. 표준화가 한 줄이 된다

In [ ]:
mu = X.mean(dim=0)                      # (4,)
sd = X.std(dim=0, unbiased=False)       # (4,)
Z = (X - mu) / sd                       # (333,4) - (4,) → 브로드캐스팅

print('X  :', X.shape, '  mu :', mu.shape)
print('Z  :', Z.shape)
print('\nZ의 변수별 평균    :', Z.mean(dim=0).numpy().round(4))
print('Z의 변수별 표준편차:', Z.std(dim=0, unbiased=False).numpy().round(4))

**333 × 4 개의 값이 반복문 없이 한 줄로** 처리되었다.

## 4-2. 실패하는 경우

In [ ]:
try:
    _ = M + torch.tensor([1.0, 2.0])     # (2,3) + (2,) → 뒤에서부터 3 vs 2
except RuntimeError as e:
    print('에러 :', str(e)[:90])

print('\n해결 : (2,) 를 (2, 1) 로 세우면 열 방향으로 퍼진다')
print(M + torch.tensor([1.0, 2.0]).unsqueeze(1))

> **직접 해보기 ④ — Min-Max 정규화**
>
>
> 각 변수를 **0~1 범위**로 바꾼 `Xn` 을 브로드캐스팅으로 만드시오.
>
> $$x' = \frac{x - \min}{\max - \min}$$

In [ ]:
# ✏️ 직접 채워 보세요
Xn = None            # ← 여기를 채우세요

assert Xn is not None and Xn.shape == X.shape
assert abs(float(Xn.min())) < 1e-5 and abs(float(Xn.max()) - 1.0) < 1e-5
print('통과  최소', float(Xn.min()), ' 최대', float(Xn.max()))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
mn = X.min(dim=0).values
mx = X.max(dim=0).values
Xn = (X - mn) / (mx - mn)
print('변수별 최소:', Xn.min(dim=0).values.numpy())
print('변수별 최대:', Xn.max(dim=0).values.numpy())

---

# 5. 여러 종류의 데이터를 행렬로 만든다

이 수업은 세 종류의 데이터를 다룬다. **전부 결국 숫자 행렬이 된다.**

## 5-1. 표 — 숫자 열은 그대로

In [ ]:
X = torch.tensor(df[cols].to_numpy(dtype='float32'))
print('표 →', X.shape, '  (n, p)')

## 5-2. 표 — 글자 열은 숫자로

`species` 열은 글자다. 그대로는 행렬에 넣을 수 없다.

In [ ]:
print('원래 값 :', df['species'].unique().tolist())

codes = pd.Categorical(df['species'], categories=['Adelie', 'Chinstrap', 'Gentoo']).codes
y_species = torch.tensor(codes.astype('int64'))

print('숫자로   :', y_species[:8].tolist(), '...')
print('shape    :', y_species.shape, '  dtype', y_species.dtype)
print('클래스별 수:', torch.bincount(y_species).tolist())

> **번호에 크기의 뜻은 없다**
>
>
> Adelie=0, Chinstrap=1, Gentoo=2 는 **이름표**일 뿐이다.
> "Gentoo가 Adelie의 2배"가 아니다. 이 번호를 **입력**으로 쓰려면 다른 처리(원-핫)가 필요한데,
> 4주차에서 다룬다. 지금은 **정답(클래스 번호)** 으로만 쓴다.


## 5-3. 사진 — 픽셀이 곧 행렬이다 (5주차 예고)

In [ ]:
import os, urllib.request
from PIL import Image

os.makedirs('./data', exist_ok=True)
IMG = './data/dog1.jpg'
if not os.path.exists(IMG):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/pytorch/vision/main/gallery/assets/dog1.jpg', IMG)

img = Image.open(IMG).convert('RGB')
print('원본 크기 :', img.size)

In [ ]:
gray8 = np.array(img.resize((8, 8)).convert('L'))     # 8x8 흑백으로 줄여 본다
print('배열 shape :', gray8.shape, '  dtype', gray8.dtype)
print(gray8)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(img); axes[0].set_title('original', fontsize=9)
axes[1].imshow(gray8, cmap='gray'); axes[1].set_title('8x8 gray', fontsize=9)
axes[2].imshow(np.array(img.resize((64, 64)))); axes[2].set_title('64x64 color', fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
color = np.array(img.resize((64, 64)))
print('흑백 :', gray8.shape, '        ← (높이, 너비)')
print('컬러 :', color.shape, ' ← (높이, 너비, 채널)  채널 = R, G, B')
print('\n값의 범위 :', color.min(), '~', color.max(), ' (0~255 정수)')
print('텐서로    :', torch.tensor(color, dtype=torch.float32).shape)

**사진은 그냥 숫자 격자다.** 밝은 곳이 큰 수, 어두운 곳이 작은 수다. 5주차의 시작점이 여기다.

## 5-4. 문장 — 단어를 번호로 (10주차 예고)

In [ ]:
sentences = ['딥러닝은 데이터로 규칙을 배운다',
             '기계학습은 데이터로 규칙을 배운다',
             '자동차는 도로를 달린다']

words = sorted({w for s in sentences for w in s.split()})
vocab = {w: i for i, w in enumerate(words)}
print('어휘 사전 :', vocab)

In [ ]:
for s in sentences:
    ids = [vocab[w] for w in s.split()]
    print(f'{s:28s} → {ids}')

In [ ]:
# 길이가 다르면 0으로 채워 직사각형으로 만든다 (패딩)
seqs = [[vocab[w] for w in s.split()] for s in sentences]
T = max(len(q) for q in seqs)
padded = torch.zeros(len(seqs), T, dtype=torch.long)
for i, q in enumerate(seqs):
    padded[i, :len(q)] = torch.tensor(q)

print('문장 →', padded.shape, '  (문장 수, 길이)')
print(padded)

**행렬은 직사각형이어야 한다.** 문장 길이가 제각각이면 짧은 쪽을 채워 맞춘다.
10주차에서 이 문제를 제대로 다룬다.

> **오늘의 결론 하나**
>
>
> $$\text{표} \to (n, p) \qquad \text{사진} \to (H, W, C) \qquad \text{문장} \to (B, T)$$
>
> **모양만 다를 뿐 전부 숫자 텐서다.**
> 그래서 표를 다루던 도구가 사진에도, 문장에도 그대로 쓰인다.


> **직접 해보기 ⑤ — 내 문장을 행렬로**
>
>
> 문장 두 개를 직접 적고, 어휘 사전을 만들어 패딩된 행렬로 만드시오.

In [ ]:
# ✏️ 직접 채워 보세요
my_sentences = [None, None]        # ← 문장 두 개를 적으세요

# 어휘 사전을 만들고, 번호로 바꾸고, 패딩하세요
my_padded = None

assert my_padded is not None and my_padded.dim() == 2
print('통과', my_padded.shape)
print(my_padded)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
my_sentences = ['오늘 점심은 김치찌개', '내일 점심은 짜장면 아니면 짬뽕']

my_words = sorted({w for s in my_sentences for w in s.split()})
my_vocab = {w: i + 1 for i, w in enumerate(my_words)}      # 0은 패딩용으로 비워 둔다

my_seqs = [[my_vocab[w] for w in s.split()] for s in my_sentences]
T2 = max(len(q) for q in my_seqs)
my_padded = torch.zeros(len(my_seqs), T2, dtype=torch.long)
for i, q in enumerate(my_seqs):
    my_padded[i, :len(q)] = torch.tensor(q)

print(my_vocab)
print(my_padded)

---

# 6. 완성 — 행렬 곱 한 번이 곧 예측이다

Ch01의 모형을 다시 본다.

$$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + b$$

## 6-1. 한 마리를 예측하면 — 내적

In [ ]:
w = torch.tensor([4.0, 10.0, 50.0, 0.0])
b = -6000.0

x0 = X[0]
pred0 = (w * x0).sum() + b            # 곱해서 더한다 = 내적

print('입력 :', x0.numpy())
print('가중치:', w.numpy())
print('예측 :', round(float(pred0), 1), 'g')
print('실제 :', float(y[0]), 'g')

## 6-2. 333마리를 한 번에 — 행렬 곱

In [ ]:
yhat = X @ w + b

print('X     :', X.shape)
print('w     :', w.shape)
print('X @ w :', (X @ w).shape, '  ← 반복문 없이 333개가 한 번에')
print('\n앞 5마리 예측:', yhat[:5].numpy().round(1))
print('앞 5마리 실제:', y[:5].numpy())

In [ ]:
loop = torch.tensor([float((w * X[i]).sum() + b) for i in range(len(X))])
print('반복문과 같은가:', torch.allclose(loop, yhat, atol=1e-2))

> **행렬 곱의 모양 규칙**
>
>
> $$(n,\ \color{red}{p}) \;@\; (\color{red}{p},) \;\longrightarrow\; (n,)$$
>
> **안쪽 두 숫자가 같아야** 곱해진다. 이 규칙 하나로 대부분의 shape 에러가 설명된다.

In [ ]:
try:
    _ = X @ torch.tensor([1.0, 2.0, 3.0])       # (333,4) @ (3,)
except RuntimeError as e:
    print('에러 :', str(e)[:100])

## 6-3. 얼마나 틀렸나 — 손실

In [ ]:
def rmse(pred, target):
    return float(((pred - target) ** 2).mean().sqrt())

print('RMSE :', round(rmse(yhat, y), 1), 'g')
print('몸무게 표준편차:', round(float(y.std(unbiased=False)), 1), 'g  ← 아무것도 안 하면 이만큼 틀린다')

## 6-4. 가중치를 바꾸면 손실이 달라진다

날개 길이의 가중치 하나만 훑어 본다.

In [ ]:
candidates = torch.linspace(0, 100, 101)
losses = []
for c in candidates:
    w_try = torch.tensor([4.0, 10.0, float(c), 0.0])
    losses.append(rmse(X @ w_try + b, y))

best = float(candidates[int(np.argmin(losses))])
plt.figure(figsize=(5.8, 3.5))
plt.plot(candidates, losses)
plt.axvline(best, color='red', ls='--', label=f'best = {best:.0f}')
plt.xlabel('weight of flipper_length'); plt.ylabel('RMSE (g)')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print('가장 좋은 값:', best, '  그때 RMSE:', round(min(losses), 1))

**곡선의 바닥을 찾는 것** — 이것이 Ch01에서 말한 **학습**이다.
가중치가 4개면 4차원이고, 신경망에서는 수백만 차원이 된다.
어떻게 찾는지는 **3주차**에서 배운다.

> **직접 해보기 ⑥ — 다른 가중치를 훑어 보기**
>
>
> 부리 길이의 가중치(`w[0]`)를 0~100으로 훑어 최적값을 찾으시오.
> 날개 길이만큼 손실이 크게 줄어드는가?

In [ ]:
# ✏️ 직접 채워 보세요
losses2 = []
for c in torch.linspace(0, 100, 101):
    w_try = None                # ← w[0]만 c로 바꾼 가중치
    losses2.append(rmse(X @ w_try + b, y))

print('최소 RMSE :', round(min(losses2), 1))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
cands = torch.linspace(0, 100, 101)
losses2 = []
for c in cands:
    w_try = torch.tensor([float(c), 10.0, 50.0, 0.0])
    losses2.append(rmse(X @ w_try + b, y))

print(f'부리 길이 : 최적 {float(cands[int(np.argmin(losses2))]):.0f}   RMSE {min(losses2):.1f} g')
print(f'날개 길이 : 최적 {best:.0f}   RMSE {min(losses):.1f} g')

부리 길이의 최적 가중치가 **0**으로 나왔다.
날개 길이(50)가 이미 몸무게를 잘 설명하고 있어서, 부리 길이를 더해도 나아지는 것이 없다는 뜻이다.

> **변수의 가치는 혼자서가 아니라 다른 변수와 함께 정해진다.**
> "부리 길이는 쓸모없다"가 아니라 "날개 길이가 있는 한 부리 길이는 더 보탤 것이 없다"이다.
> 날개 길이의 가중치를 0으로 고정하고 다시 훑어 보면 결과가 달라진다.

In [ ]:
wide = torch.linspace(0, 120, 121)

for idx, name in [(0, '부리 길이'), (2, '날개 길이')]:
    L = []
    for c in wide:
        w_try = torch.zeros(4)
        w_try[idx] = float(c)                 # 그 변수 하나만, 편향도 0
        L.append(rmse(X @ w_try, y))
    i = int(np.argmin(L))
    print(f'{name}만 : 최적 w={float(wide[i]):5.0f}   RMSE {L[i]:7.1f} g')

print(f'{"아무것도 안 쓰면":8s} : {float(y.std(unbiased=False)):23.1f} g')

혼자 쓰면 **날개 길이(566g)가 부리 길이(652g)보다 낫다.**
둘 다 "아무것도 안 하기(804g)"보다는 낫다.

---

# 7. 정리

> **이번 주 체크포인트**
>
>
> | 하고 싶은 일 | 코드 | 결과 shape |
> |------|------|------|
> | 표 → 행렬 | `torch.tensor(df[cols].to_numpy(dtype='float32'))` | `(n, p)` |
> | 한 데이터 포인트 | `X[i]` | `(p,)` |
> | 한 변수 전체 | `X[:, j]` | `(n,)` |
> | 조건으로 고르기 | `X[X[:, 3] > 5000]` | `(선택 수, p)` |
> | **변수별** 통계 | `X.mean(dim=0)` | `(p,)` |
> | 데이터별 통계 | `X.mean(dim=1)` | `(n,)` |
> | 세로로 세우기 | `y.unsqueeze(1)` | `(n, 1)` |
> | 표준화 (브로드캐스팅) | `(X - X.mean(0)) / X.std(0)` | `(n, p)` |
> | 글자 → 번호 | `pd.Categorical(col, categories=[...]).codes` | `(n,)` |
> | 사진 → 배열 | `np.array(img)` | `(H, W)` 또는 `(H, W, C)` |
> | 문장 → 번호 | 어휘 사전 + 패딩 | `(B, T)` |
> | **예측 한 번에** | `X @ w + b` | `(n,)` |


**오늘의 결론**

$$\text{표} \to (n, p), \quad \text{사진} \to (H, W, C), \quad \text{문장} \to (B, T)$$

무엇이 들어오든 **숫자 텐서**가 되고, **행렬 곱 한 번**이 예측이 된다.
막히면 **`.shape` 을 찍는다.**

## 스스로 확인해 보기

아래 결과를 먼저 예상한 뒤 실행해서 대조한다.

In [ ]:
A = torch.tensor([[0., 1., 2., 3.],
                  [4., 5., 6., 7.],
                  [8., 9., 10., 11.]])
v = torch.tensor([1.0, 0.0, 2.0, 0.0])

print('A =\n', A)
print('\nA.shape          :', A.shape)
print('A[1]             :', A[1])
print('A[:, 2]          :', A[:, 2])
print('A.sum(dim=0)     :', A.sum(dim=0))
print('A.sum(dim=1)     :', A.sum(dim=1))
print('(A + v).shape    :', (A + v).shape)
print('A @ v            :', A @ v, ' shape', (A @ v).shape)

---

## 다음 실습

[실습 2주차: 퍼셉트론을 쌓아 신경망 만들기](lab02.qmd) —
오늘의 `X @ w + b` 가 곧 **퍼셉트론 하나**다.
그것을 여러 개, 여러 층으로 쌓으면 직선으로 안 되던 문제가 풀리기 시작한다.